In [ ]:
# Nguyễn Hoàng Tùng

import pandas as pd

ratings = pd.read_csv('ratings.csv')  # userId, itemId, rating
reviews = pd.read_csv('reviews.csv')  # userId, itemId, review_text
products = pd.read_csv('products.csv')  # itemId, category, price, description

print(ratings.head())
print(products.sample(3))


In [ ]:
# Nguyễn Hoàng Tùng
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

ratings['rating_norm'] = (ratings['rating'] - ratings['rating'].min()) / (ratings['rating'].max() - ratings['rating'].min())
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
X_text = tfidf.fit_transform(reviews['review_text'])


In [ ]:
# Nguyễn Hoàng Tùng

import tensorflow as tf
from tensorflow.keras import layers, Model

num_users = ratings['userId'].nunique()
num_items = ratings['itemId'].nunique()

user_input = layers.Input(shape=(1,))
item_input = layers.Input(shape=(1,))

user_vec = layers.Embedding(num_users, 64)(user_input)
item_vec = layers.Embedding(num_items, 64)(item_input)

dot = layers.Dot(axes=2)([user_vec, item_vec])
output = layers.Dense(1, activation='sigmoid')(dot)

model = Model([user_input, item_input], output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Nguyễn Hoàng Tùng

history = model.fit(
    [train.userId, train.itemId],
    train.rating_norm,
    validation_data=([test.userId, test.itemId], test.rating_norm),
    epochs=10,
    batch_size=256


In [ ]:
# Nguyễn Hoàng Tùng

from tensorflow.keras.callbacks import EarlyStopping

callback = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
model.fit(..., callbacks=[callback])


In [ ]:
# Nguyễn Hoàng Tùng

from fastapi import FastAPI
import joblib

app = FastAPI()
model = tf.keras.models.load_model('recommender_model.h5')

@app.post('/recommend')
def recommend(user_id: int):
    items = list(range(num_items))
    preds = model.predict([ [user_id]*len(items), items ])
    top_items = sorted(zip(items, preds.flatten()), key=lambda x: -x[1])[:10]
    return {"user_id": user_id, "recommendations": top_items}
